In [3]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import anthropic
import gradio as gr

In [4]:
# Models

load_dotenv(override=True)

MODEL_CLAUDE = 'claude-haiku-4-5'
MODEL_LLAMA = 'llama3.2'

# Claude

claude = anthropic.Anthropic()

In [31]:
# Tools

tools = []

In [32]:
# Tooling Functions - Ticket Prices

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown")
    return f"The price of a ticket to {destination_city} is {price}."

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a ticket to the destination city.",
    "input_schema": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city the customer wants to travel to.",
            },
        },
        "required": ["destination_city"],
    }
}

tools.append(price_function)

# Tooling Function = Set Ticket Prices

def set_ticket_price(destination_city, price):
    print(f"Tool called to set price of {price} for {destination_city}.")
    ticket_prices[destination_city.lower()] = price
    return f"Price for a ticket to {destination_city} has been set to {price}."

set_price_function = {
    "name": "set_ticket_price",
    "description": "Sets the price of a ticket to the destination city.",
    "input_schema": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city the customer wants to change the price of.",
            },
            "price": {
                "type": "string",
                "description": "The price that the destination city is being set to."
                }
                },
        "required": ["destination_city", "price"],
    }
}

tools.append(set_price_function)
    

In [28]:
# Handle Tool Calls

def handle_tool_calls(response):

    tool_results = []
    for block in response.content:
        if block.type == "tool_use":
            if block.name == "get_ticket_price":
                city = block.input.get("destination_city")
                result = get_ticket_price(city)
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": result})
            elif block.name == "set_ticket_price":
                city = block.input.get("destination_city")
                price = block.input.get("price")
                result = set_ticket_price(city, price)
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": result})

    return tool_results

In [30]:
# System Message

system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

# Chat Function

def chat(message, history): # message -> user input, history -> all past messages
    history = [{"role": h["role"], "content": h["content"]} for h in history] 
    messages = history + [{"role": "user", "content": message}]
    response = claude.messages.create(
         model='claude-haiku-4-5',
         max_tokens=200,
         system=system_message,
         messages=messages,
         tools = tools)

    while response.stop_reason == "tool_use":

        # add tool call to history
        messages.append({"role": "assistant", "content": response.content})
        tool_results = handle_tool_calls(response)
        messages.append({"role": "user", "content": tool_results})

        response = claude.messages.create(
          model='claude-haiku-4-5',
          max_tokens=200,
          system=system_message,
          messages=messages,
          tools = tools)

    return next(block.text for block in response.content if block.type == "text")

In [33]:
# Gradio

gr.ChatInterface(fn=chat, type='messages').launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


Tool called for London
Tool called to set price of $599 for London.
Tool called for london
Tool called to set price of 399 for London.
Tool called for London
